In [48]:
import sqlite3
import hashlib
import os
import subprocess
import random
import pickle

print('Imports loaded')

Imports loaded


In [49]:

DB_USER     = "admin"
DB_PASSWORD = "admin123"
SECRET_KEY  = "supersecretkey123"

print(f'DB_USER     = {DB_USER}')
print(f'DB_PASSWORD = {DB_PASSWORD}')
print(f'SECRET_KEY  = {SECRET_KEY}')
print()
print('These credentials are exposed to anyone who reads this file!')

DB_USER     = admin
DB_PASSWORD = admin123
SECRET_KEY  = supersecretkey123

These credentials are exposed to anyone who reads this file!


In [50]:
def get_connection():
    return sqlite3.connect(":memory:")  


CONN = get_connection()

def setup_db():
    CONN.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY,
            username TEXT,
            password TEXT,
            email TEXT,
            role TEXT
        )
    """)
    CONN.commit()
    print('Database created')

setup_db()

Database created


In [51]:

def login_vulnerable(username, password):
    
    query = "SELECT * FROM users WHERE username = '" + username + "' AND password = '" + password + "'"
    print(f'SQL Query built: {query}')
    result = CONN.execute(query).fetchone()
    return result


print('=== Normal login ===')
login_vulnerable('alice', 'pass123')

print()
print('=== SQL Injection Attack ===')

login_vulnerable("' OR '1'='1' --", "anything")
print()
print("  The injected query returns ALL users, bypassing the password check!")

=== Normal login ===
SQL Query built: SELECT * FROM users WHERE username = 'alice' AND password = 'pass123'

=== SQL Injection Attack ===
SQL Query built: SELECT * FROM users WHERE username = '' OR '1'='1' --' AND password = 'anything'

  The injected query returns ALL users, bypassing the password check!


In [52]:

def hash_password_vulnerable(password):
    return hashlib.md5(password.encode()).hexdigest()

pwd = 'password123'
hashed = hash_password_vulnerable(pwd)
print(f'Password : {pwd}')
print(f'MD5 Hash : {hashed}')
print()
print('This exact hash (482c811da5d5b4bc6d497ffa98491e38) is in every rainbow table online!')
print('Try it yourself: https://crackstation.net')

Password : password123
MD5 Hash : 482c811da5d5b4bc6d497ffa98491e38

This exact hash (482c811da5d5b4bc6d497ffa98491e38) is in every rainbow table online!
Try it yourself: https://crackstation.net


In [53]:

def register_user_vulnerable(username, password, email, role="user"):
   
    hashed = hash_password_vulnerable(password)
    CONN.execute(
        "INSERT INTO users (username, password, email, role) VALUES (?, ?, ?, ?)",
        (username, hashed, email, role)
    )
    CONN.commit()
   
    print(f'User {username} registered with password hash: {hashed}')


register_user_vulnerable('alice', 'password123', 'alice@example.com')

print()

register_user_vulnerable('hacker', 'abc', '', role='admin')
print()
print('Empty email and weak password accepted — no validation at all!')
print('Anyone can self-assign role=admin!')

User alice registered with password hash: 482c811da5d5b4bc6d497ffa98491e38

User hacker registered with password hash: 900150983cd24fb0d6963f7d28e17f72

Empty email and weak password accepted — no validation at all!
Anyone can self-assign role=admin!


In [54]:

def ping_host_vulnerable(host):
    command = "ping -c 1 " + host   
    print(f'Command that would run: {command}')
   

print('=== Normal input ===')
ping_host_vulnerable('8.8.8.8')

print()
print('=== Command Injection Attack ===')
ping_host_vulnerable('8.8.8.8; cat /etc/passwd')
print()
print('The second command would execute cat /etc/passwd on the server!')

=== Normal input ===
Command that would run: ping -c 1 8.8.8.8

=== Command Injection Attack ===
Command that would run: ping -c 1 8.8.8.8; cat /etc/passwd

The second command would execute cat /etc/passwd on the server!


In [55]:

def read_user_file_vulnerable(filename):
    base_dir = "/var/app/uploads/"
    filepath = base_dir + filename 
    print(f'Would open file: {filepath}')
   

print('=== Normal filename ===')
read_user_file_vulnerable('profile_picture.jpg')

print()
print('=== Path Traversal Attack ===')
read_user_file_vulnerable('../../etc/passwd')
print()
print('Attacker can read ANY file on the server!')

=== Normal filename ===
Would open file: /var/app/uploads/profile_picture.jpg

=== Path Traversal Attack ===
Would open file: /var/app/uploads/../../etc/passwd

Attacker can read ANY file on the server!


In [56]:

def generate_token_vulnerable():
    return str(random.randint(100000, 999999))   

print('Generating 5 sample tokens (insecure):')
for i in range(5):
    print(f'  Token {i+1}: {generate_token_vulnerable()}')

print()
print('Only 900,000 possible values — an attacker can brute-force this in seconds!')
print('random module is NOT cryptographically secure')

Generating 5 sample tokens (insecure):
  Token 1: 737176
  Token 2: 221464
  Token 3: 232016
  Token 4: 609211
  Token 5: 250730

Only 900,000 possible values — an attacker can brute-force this in seconds!
random module is NOT cryptographically secure


In [57]:

def get_user_vulnerable(user_id):
    try:
        result = CONN.execute(f"SELECT * FROM users WHERE id = {user_id}").fetchone()
        return result
    except Exception as e:
        return f"Database error: {str(e)}"   


result = get_user_vulnerable("' OR 1=1 --")
print(f'Error returned to user: {result}')
print()
print('Raw error messages reveal database structure, table names, and server paths to attackers!')

Error returned to user: Database error: unrecognized token: "' OR 1=1 --"

Raw error messages reveal database structure, table names, and server paths to attackers!


In [58]:

def load_session_vulnerable(session_data):
    return pickle.loads(session_data)  


safe_data = pickle.dumps({'user_id': 1, 'role': 'user'})
result = load_session_vulnerable(safe_data)
print(f'Legitimate session: {result}')

print()
print('If an attacker crafts a malicious pickle payload, pickle.loads() will execute it!')
print('This is a Remote Code Execution (RCE) vulnerability — one of the most critical!')

Legitimate session: {'user_id': 1, 'role': 'user'}

If an attacker crafts a malicious pickle payload, pickle.loads() will execute it!
This is a Remote Code Execution (RCE) vulnerability — one of the most critical!


In [59]:
vulnerabilities = [
    ('VUL-01', 'SQL Injection',                 'CRITICAL', 'CWE-89'),
    ('VUL-02', 'Hardcoded Credentials',          'CRITICAL', 'CWE-798'),
    ('VUL-03', 'Command Injection',              'CRITICAL', 'CWE-78'),
    ('VUL-04', 'Insecure Deserialization',        'CRITICAL', 'CWE-502'),
    ('VUL-05', 'Weak Hashing (MD5)',              'HIGH',     'CWE-916'),
    ('VUL-06', 'Path Traversal',                 'HIGH',     'CWE-22'),
    ('VUL-07', 'Insecure Random Token',          'HIGH',     'CWE-338'),
    ('VUL-08', 'Sensitive Data in Logs',         'HIGH',     'CWE-532'),
    ('VUL-09', 'Missing Input Validation',       'MEDIUM',   'CWE-20'),
    ('VUL-10', 'Verbose Error Messages',         'MEDIUM',   'CWE-209'),
]

severity_icon = {'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡'}

print('=' * 65)
print('  CodeAlpha Task 3 — Vulnerability Audit Summary')
print('=' * 65)
print(f'{"ID":<10}{"Vulnerability":<32}{"Severity":<12}{"CWE"}')
print('-' * 65)
for vid, name, sev, cwe in vulnerabilities:
    icon = severity_icon[sev]
    print(f'{vid:<10}{name:<32}{icon} {sev:<10}{cwe}')
print('=' * 65)

counts = {}
for _, _, sev, _ in vulnerabilities:
    counts[sev] = counts.get(sev, 0) + 1

print(f'\n  Total: {len(vulnerabilities)} vulnerabilities found')
for sev, count in counts.items():
    print(f'  {severity_icon[sev]} {sev}: {count}')
print()
print('  → See vulnerable_app.ipynb vs secure_app.ipynb for all fixes')

  CodeAlpha Task 3 — Vulnerability Audit Summary
ID        Vulnerability                   Severity    CWE
-----------------------------------------------------------------
VUL-01    SQL Injection                   🔴 CRITICAL  CWE-89
VUL-02    Hardcoded Credentials           🔴 CRITICAL  CWE-798
VUL-03    Command Injection               🔴 CRITICAL  CWE-78
VUL-04    Insecure Deserialization        🔴 CRITICAL  CWE-502
VUL-05    Weak Hashing (MD5)              🟠 HIGH      CWE-916
VUL-06    Path Traversal                  🟠 HIGH      CWE-22
VUL-07    Insecure Random Token           🟠 HIGH      CWE-338
VUL-08    Sensitive Data in Logs          🟠 HIGH      CWE-532
VUL-09    Missing Input Validation        🟡 MEDIUM    CWE-20
VUL-10    Verbose Error Messages          🟡 MEDIUM    CWE-209

  Total: 10 vulnerabilities found
  🔴 CRITICAL: 4
  🟠 HIGH: 4
  🟡 MEDIUM: 2

  → See vulnerable_app.ipynb vs secure_app.ipynb for all fixes
